# Apps from Models - Nextmv & Gurobi

This notebook shows how to:

1. Create a decision model that solves a knapsack problem with the
   `nextmv-gurobipy` package.
2. Run it locally.
3. Push it to a Nextmv Cloud Application.
4. Run it remotely.

Let’s dive right in! 🐰

# Dependencies

Install the necessary Python packages.



In [ ]:
%pip install nextmv-gurobipy

# Imports

Add the necessary imports.

In [ ]:
import json
import os
import time

import nextmv
import nextmv.cloud
import nextmv_gurobipy as ngp
from gurobipy import GRB

# 1. Create the decision model

Use Gurobi to solve a classic MIP with the `nextmv.Model` class.

In [ ]:
class DecisionModel(nextmv.Model):
    def solve(self, input: nextmv.Input) -> nextmv.Output:
        """Solves the given problem and returns the solution."""

        start_time = time.time()
        model = ngp.Model(input.options)

        # Initializes the linear sums.
        weights = 0.0
        values = 0.0

        # Creates the decision variables and adds them to the linear sums.
        items = []
        for item in input.data["items"]:
            item_variable = model.addVar(vtype=GRB.BINARY, name=item["id"])
            items.append({"item": item, "variable": item_variable})
            weights += item_variable * item["weight"]
            values += item_variable * item["value"]

        # This constraint ensures the weight capacity of the knapsack will not be
        # exceeded.
        model.addConstr(weights <= input.data["weight_capacity"])

        # Sets the objective function: maximize the value of the chosen items.
        model.setObjective(expr=values, sense=GRB.MAXIMIZE)

        # Solves the problem.
        model.optimize()

        return nextmv.Output(
            options=input.options,
            solution=ngp.ModelSolution(model),
            statistics=ngp.ModelStatistics(model, start_time),
        )


# 2. Run the model locally

Define the options that the model needs.

In [ ]:
options = ngp.ModelOptions().to_nextmv()

Instantiate the model.

In [ ]:
model = DecisionModel()

Define some sample input data.

In [ ]:
sample_input = {
  "items": [
    {
      "id": "cat",
      "value": 100,
      "weight": 20
    },
    {
      "id": "dog",
      "value": 20,
      "weight": 45
    },
    {
      "id": "water",
      "value": 40,
      "weight": 2
    },
    {
      "id": "phone",
      "value": 6,
      "weight": 1
    },
    {
      "id": "book",
      "value": 63,
      "weight": 10
    },
    {
      "id": "rx",
      "value": 81,
      "weight": 1
    },
    {
      "id": "tablet",
      "value": 28,
      "weight": 8
    },
    {
      "id": "coat",
      "value": 44,
      "weight": 9
    },
    {
      "id": "laptop",
      "value": 51,
      "weight": 13
    },
    {
      "id": "keys",
      "value": 92,
      "weight": 1
    },
    {
      "id": "nuts",
      "value": 18,
      "weight": 4
    }
  ],
  "weight_capacity": 50
}

Run the model locally. First, write the file with the license info, then solve the model.

In [ ]:
%%writefile gurobi.lic
WLSACCESSID=REPLACE_ME
WLSSECRET=REPLACE_ME
LICENSEID=REPLACE_ME

In [ ]:
input = nextmv.Input(data=sample_input, options=options)
output = model.solve(input)
print(json.dumps(output.solution, indent=2))

# 3. Push the model to Nextmv Cloud

Convert the model to an application, hence the workflow name "Apps from
Models". Push the application to Nextmv Cloud.

Every app is production-ready with a full-featured API.

In [ ]:
client = nextmv.cloud.Client(api_key=os.getenv("NEXTMV_API_KEY"))
application = nextmv.cloud.Application(client=client, id="apps-from-models-gurobi")

model_configuration = nextmv.ModelConfiguration(
    name="gurobi_model",
    requirements=[
        "nextmv-gurobipy==0.4.1",
    ],
    options=options,
)
manifest = nextmv.cloud.Manifest.from_model_configuration(model_configuration)
manifest.files.append("gurobi.lic")

application.push(
    model=model,
    model_configuration=model_configuration,
    manifest=manifest,
    verbose=True,
)

# 4. Run the model remotely

Execute an app run. This remote run produces an output that should be the same as the local run.

In [ ]:
result = application.new_run_with_result(input=sample_input, instance_id="devint")
print(json.dumps(result.output, indent=2))